# Pz3 Abacus Paste Runtime Scaling Diagnostics

This notebook summarizes exact sampled-galaxy GPU chunk benchmarks for the pz3 600 deg2 Abacus Backlight validation. It intentionally excludes the rejected weighted-HOD path.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = Path('/mnt/ceph/users/spandey/ltu-godmax/GODMAX')
meas = root / 'data/xDESI/processed/abacus_backlight/stage31_pz3_cap600/measurements'
files = sorted(meas.glob('gpu_chunk_scaling_nside1024_halos*_job*.json'))

rows = []
for path in files:
    payload = json.loads(path.read_text())
    if not payload.get('use_multi_kappa_maps'):
        continue
    timing_rows = payload.get('rows', [])
    fused = [row for row in timing_rows if row.get('fused') is True]
    if not fused:
        continue
    row = fused[-1]
    tr = row.get('timing_results', {})
    rows.append({
        'file': path.name,
        'n_halos': int(payload['n_halos']),
        'pixel_time_s': float(payload.get('pixel_time_s', np.nan)),
        'runtime_s': float(row.get('runtime_s', np.nan)),
        'galaxy_population_s': float(tr.get('galaxy_population', np.nan)),
        'ymap_s': float(tr.get('ymap_generation_and_assembly', np.nan)),
        'ksz_tau_s': float(tr.get('ksz_tau_map_generation_and_assembly', np.nan)),
        'multi_kappa_s': float(tr.get('multi_kappa_map_generation_and_assembly', np.nan)),
        'n_pairs': int(payload.get('n_pairs', -1)),
    })

df = pd.DataFrame(rows).sort_values(['n_halos', 'runtime_s']).reset_index(drop=True)
df

In [ ]:
best = df.loc[df.groupby('n_halos')['runtime_s'].idxmin()].sort_values('n_halos')
display(best[['n_halos', 'pixel_time_s', 'runtime_s', 'galaxy_population_s', 'n_pairs', 'file']])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
ax = axes[0]
ax.plot(best['n_halos'] / 1e6, best['runtime_s'], 'o-', label='GPU map + exact sampled galaxies')
ax.plot(best['n_halos'] / 1e6, best['galaxy_population_s'], 's-', label='Exact HOD population only')
ax.plot(best['n_halos'] / 1e6, best['pixel_time_s'], '^-', label='CPU pixel work package')
ax.set_xlabel('Halos in benchmark chunk [millions]')
ax.set_ylabel('Wall time [s]')
ax.set_title('Single A100 exact chunk scaling')
ax.grid(alpha=0.25)
ax.legend(frameon=False)

ax = axes[1]
components = ['ymap_s', 'ksz_tau_s', 'multi_kappa_s', 'galaxy_population_s']
bottom = np.zeros(len(best))
for comp in components:
    vals = best[comp].fillna(0.0).to_numpy()
    ax.bar(best['n_halos'] / 1e6, vals, bottom=bottom, width=0.06, label=comp.replace('_s', ''))
    bottom += vals
ax.set_xlabel('Halos in benchmark chunk [millions]')
ax.set_ylabel('Component time [s]')
ax.set_title('Fused exact runtime breakdown')
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False, fontsize=9)
plt.show()

In [ ]:
# Compare grouped exact HOD against the best ungrouped exact 600k run.
df600 = df[df['n_halos'] == 600000].sort_values('runtime_s')
display(df600[['runtime_s', 'galaxy_population_s', 'pixel_time_s', 'file']])
print('Use the fastest exact 600k/1M rows above for production. Grouped max_gals is a regression if its row is slower.')